## Method C — Recursive Feature Elimination (RFE)

**What this notebook does**

1. Fit **RFE** on **training data only** with a base estimator (default: logistic regression). 
    - RFE repeatedly trains the estimator and drops the weakest feature until `n_features_to_select` remain.
2. Show which features are kept (`support_`) and their **`ranking_`** (1 = survived in the final set).
3. Save a small CSV under `src/feature_selection/`.
4. Compare **all features** vs **RFE-selected features** on the test set using the same Random Forest settings as `baseline_final`.

Find the repo root and add `src` to `sys.path` so `stroke_data` imports regardless of kernel cwd.


In [1]:
import sys
from pathlib import Path

_here = Path().resolve()
REPO_ROOT = _here
while REPO_ROOT != REPO_ROOT.parent:
    if (REPO_ROOT / "src" / "stroke_data.py").is_file():
        break
    REPO_ROOT = REPO_ROOT.parent
else:
    raise FileNotFoundError("Could not find src/stroke_data.py (open this project from the repo folder).")

sys.path.insert(0, str(REPO_ROOT / "src"))

Imports include **`LogisticRegression`** (RFE inner model), `RFE`, `RandomForestClassifier` (evaluation later), metrics, and `stroke_data`. Also set CSV path, output path, `N_FEATURES_TO_SELECT`, and `RFE_STEP`.


In [2]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

from stroke_data import get_stroke_data_for_cv

CSV = "data/knn-standardize-distance.csv"
OUT = "src/feature_selection/rfe_selection.csv"
N_FEATURES_TO_SELECT = 8
RFE_STEP = 1

Load data: feature names from the CSV; train/test split via `stroke_data`.


In [3]:
csv_path = REPO_ROOT / CSV
df = pd.read_csv(csv_path)
feature_names = [c for c in df.columns if c not in ("id", "stroke")]

X_train, X_test, y_train, y_test = get_stroke_data_for_cv(str(csv_path))
len(feature_names), X_train.shape

(10, (4088, 10))

### RFE fit (training data only)

Define `LogisticRegression` as the estimator RFE retrains each round, then wrap it in `RFE` and call `fit` on `X_train`, `y_train` only.


In [ ]:
base_estimator = LogisticRegression(
    random_state=42,
    class_weight="balanced",
    max_iter=2000,
    solver="lbfgs",
)

selector = RFE(
    estimator=base_estimator,
    n_features_to_select=N_FEATURES_TO_SELECT,
    step=RFE_STEP,
)
selector.fit(X_train, y_train)

### Selected features and rankings (train-only fit)

Tabulate `support_` and `ranking_` per feature.


In [5]:
rfe_df = pd.DataFrame({
    "feature": feature_names,
    "selected": selector.support_,
    "ranking": selector.ranking_,
}).sort_values("ranking")
rfe_df

,feature,selected,ranking
0,gender,True,1
1,age,True,1
2,hypertension,True,1
3,heart_disease,True,1
4,ever_married,True,1
6,Residence_type,True,1
7,avg_glucose_level,True,1
8,bmi,True,1
5,work_type,False,2
9,smoking_status,False,3


Save the table to CSV.


In [6]:
out_path = REPO_ROOT / OUT
out_path.parent.mkdir(parents=True, exist_ok=True)
rfe_df.to_csv(out_path, index=False)
print(out_path)

/Users/brad/gitClones/Stroke-Prediction-ML/src/feature_selection/rfe_selection.csv


### Test metrics — same RF as `baseline_final` (full vs RFE subset)

Evaluate the baseline Random Forest on full features vs RFE-selected columns; metrics on the test set.


In [7]:
mask = selector.support_
X_train_s = X_train[:, mask]
X_test_s = X_test[:, mask]

rf_eval = RandomForestClassifier(
    bootstrap=True,
    class_weight="balanced_subsample",
    max_depth=15,
    max_features=None,
    max_leaf_nodes=15,
    n_estimators=20,
    random_state=42,
)

rows = []
for label, Xtr, Xte in [("full", X_train, X_test), ("rfe-subset", X_train_s, X_test_s)]:
    rf_eval.fit(Xtr, y_train)
    pred = rf_eval.predict(Xte)
    rows.append({
        "tag": label,
        "accuracy": accuracy_score(y_test, pred),
        "f1": f1_score(y_test, pred, pos_label=1),
        "precision": precision_score(y_test, pred, pos_label=1, zero_division=0),
        "recall": recall_score(y_test, pred, pos_label=1),
    })

pd.DataFrame(rows).style.format({
    "accuracy": "{:.4f}",
    "f1": "{:.4f}",
    "precision": "{:.4f}",
    "recall": "{:.4f}",
}).hide(axis="index")

tag,accuracy,f1,precision,recall
full,0.7789,0.2614,0.1562,0.8000
rfe-subset,0.7730,0.2564,0.1527,0.8000
